# 03 · Processing at scale

**Story so far:** the raw review corpus sits in the object store. Sequentially looping
over it (chapter 01's naive pipeline) won't survive contact with real volume. This
chapter enriches **every review in parallel** — with an external "moderation API" in the
loop — and makes the whole thing survive failures, size skew, and slow dependencies.

**Learning goals**

1. Fan out work with `asyncio.gather` and `flyte.map.aio` — and know when to use which
2. Bound parallelism two ways: per-map `concurrency`, and cluster-wide **Union Queues**
3. Configure retries and two-dimensional timeouts (runtime vs queue wait)
4. Recover from OOM by retrying with more memory — *in code*
5. Checkpoint inside a task with `@flyte.trace`
6. Gate a pipeline on an external signal (human approval) before promoting a dataset

In [1]:
import asyncio
from typing import List

import flyte

flyte.init_from_config()

env = flyte.TaskEnvironment(
    name="processing",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

In [ ]:
# A small in-notebook sample to run the demos against. Each chapter stands alone;
# in a real engagement these rows come from chapter 02's landed corpus.
SAMPLE_REVIEWS = [
    {"id": 0, "product": "espresso machine", "stars": 5, "text": "Absolutely LOVE this espresso machine!!"},
    {"id": 1, "product": "trail shoes", "stars": 1, "text": "terrible trail shoes, arrived broken"},
    {"id": 2, "product": "headphones", "stars": 4, "text": "solid headphones, works as advertised"},
    {"id": 3, "product": "air fryer", "stars": 2, "text": "disappointed with the air fryer"},
    {"id": 4, "product": "espresso machine", "stars": 5, "text": "exceeded expectations"},
    {"id": 5, "product": "trail shoes", "stars": 3, "text": "the trail shoes are okay"},
]
len(SAMPLE_REVIEWS)

## 1. Fan-out / fan-in

In v2 there is no `@dynamic` and no special map-task construct: fan-out is plain Python.
Two idioms:

- **`asyncio.gather`** — explicit and flexible; you build the list of coroutines yourself.
  Best for heterogeneous fan-out (different tasks, conditional branches).
- **`flyte.map.aio`** — for homogeneous "same task over a list" fan-out; streams results
  as they finish, groups the actions in the UI, and has first-class partial-failure
  handling.

Our unit of work: enrich one review with a sentiment hint and a length feature.

In [2]:
@env.task
async def enrich(review: dict) -> dict:
    text = review["text"].strip().lower()
    return {
        **review,
        "text": text,
        "n_words": len(text.split()),
        "exclaims": text.count("!"),
    }


@env.task
async def enrich_gather(reviews: List[dict]) -> List[dict]:
    return list(await asyncio.gather(*[enrich(review=r) for r in reviews]))

In [3]:
@env.task
async def enrich_map(reviews: List[dict]) -> List[dict]:
    results: List[dict] = []
    # return_exceptions=True: one bad review doesn't cancel the in-flight siblings.
    # Without it, the first failure aborts the whole map.
    async for r in flyte.map.aio(enrich, reviews, return_exceptions=True):
        if isinstance(r, Exception):
            print(f"review failed: {r!r}")
            continue                      # or collect for a retry pass
        results.append(r)
    return results

In [ ]:
# Fan out: one `enrich` child action per review — watch them in the run view.
run = flyte.run(enrich_map, reviews=SAMPLE_REVIEWS)
print(run.url)
run.wait()
run.outputs()

### Binding constants with `functools.partial`

`flyte.map` iterates over **one** argument. Bind the constant ones first:

In [4]:
from functools import partial


@env.task
async def score_review(review: dict, model_version: str, threshold: int) -> bool:
    return review["stars"] <= threshold and model_version == "v2"


@env.task
async def flag_negative(reviews: List[dict]) -> List[bool]:
    fn = partial(score_review, model_version="v2", threshold=2)
    out: List[bool] = []
    async for r in flyte.map.aio(fn, reviews, return_exceptions=True):
        if isinstance(r, Exception):
            raise r
        out.append(r)
    return out

### Grouping actions in the UI

Big fan-outs make the run view noisy. `flyte.group` (or `group_name=` on `flyte.map`)
collapses related actions into a labelled group:

In [5]:
@env.task
async def staged_pipeline(reviews: List[dict]) -> List[bool]:
    with flyte.group("enrichment"):
        enriched = list(await asyncio.gather(*[enrich(review=r) for r in reviews]))

    with flyte.group("flagging"):
        fn = partial(score_review, model_version="v2", threshold=2)
        flags = list(await asyncio.gather(*[fn(review=r) for r in enriched]))

    return flags

In [ ]:
# Same fan-out, but actions collapse into "enrichment" and "flagging" groups in the UI.
run = flyte.run(staged_pipeline, reviews=SAMPLE_REVIEWS)
print(run.url)
run.wait()
run.outputs()

## 2. Bounding parallelism

An unbounded fan-out over 100k reviews will overwhelm two things long before the cluster
is the bottleneck: **your own driver** (it tracks every in-flight child) and **whatever
the reviews hit** — the rate-limited moderation API. You bound it at two levels.

**Per fan-out — `flyte.map` `concurrency`.** Cap how many child actions a *single* map
keeps in flight:

```python
async for r in flyte.map.aio(enrich, reviews, concurrency=50, return_exceptions=True):
    ...
```

That's scoped to this one map in this one run. It does nothing about a *second* run of the
same pipeline hammering the same API at the same time.

**Cluster-wide — Union Queues.** A **queue** is a control-plane scheduling lane with a
**global action-concurrency cap** (plus depth and priority). The Union engine admits at
most *N* actions from the queue at once **across every run and every user** — the exact
guarantee you want in front of a shared, rate-limited dependency. Unlike an
`asyncio.Semaphore` (which lives in one task process and is forgotten the moment the pod
restarts), the cap is enforced durably by the platform and spans concurrent runs.

Your platform team creates the queue and sets its concurrency cap, depth, and priority —
the how-to is in the [Union Queues docs](https://www.union.ai/docs/v2/union/user-guide/task-configuration/queues/); [appendix A](./appendix/A-deployment-adaptation.md)
only tracks *whether* one exists for this engagement. Tasks reference it by name with
`queue="..."`. Set it on a `TaskEnvironment`, on a single `@env.task`, at call time with
`.override(queue=...)`, or per run with `flyte.with_runcontext(queue=...)`.

In [ ]:
# Point the whole environment at a queue whose concurrency cap the platform set —
# e.g. "moderation-api" capped at 20 concurrent actions, globally.
moderation_env = flyte.TaskEnvironment(
    name="moderation",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    queue="moderation-api",              # every task here is admitted through this lane
)


@moderation_env.task
async def enrich_queued(review: dict) -> dict:
    text = review["text"].strip().lower()
    return {**review, "text": text, "n_words": len(text.split())}


@env.task
async def enrich_all(reviews: List[dict]) -> List[dict]:
    # No semaphore, no manual gate: the queue caps global concurrency for us.
    # `concurrency=` here is the optional per-map lever.
    out: List[dict] = []
    async for r in flyte.map.aio(enrich_queued, reviews, concurrency=50,
                                 return_exceptions=True):
        if isinstance(r, Exception):
            raise r
        out.append(r)
    return out


# Requires the "moderation-api" queue to exist on the deployment. Once your platform
# team has created it, this runs enrich_all through that capped lane:
# run = flyte.run(enrich_all, reviews=SAMPLE_REVIEWS)
#
# Or send ANY run through a queue at submit time, without touching definitions:
# run = flyte.with_runcontext(queue="moderation-api").run(enrich_map, reviews=SAMPLE_REVIEWS)

In [ ]:
# Per-map cap — runnable now, no queue required. At most `concurrency` enrich actions
# are in flight at once (the cluster-wide queue cap above needs the platform-defined queue).
@env.task
async def enrich_bounded(reviews: List[dict], concurrency: int = 3) -> List[dict]:
    out: List[dict] = []
    async for r in flyte.map.aio(enrich, reviews, concurrency=concurrency,
                                 return_exceptions=True):
        if isinstance(r, Exception):
            raise r
        out.append(r)
    return out


run = flyte.run(enrich_bounded, reviews=SAMPLE_REVIEWS, concurrency=3)
print(run.url)
run.wait()
run.outputs()

> **Scale note.** Every child task the driver awaits costs it some memory for tracking.
> For fan-outs beyond ~10k reviews, batch first (the micro-batching pattern in
> [05](./05-reusable-containers.ipynb)) and give the driver more memory than the workers.

## 3. Retries and timeouts

- `retries=3` → up to 4 attempts total. Applies to *task* failures (exceptions, OOM,
  node loss).
- `timeout` bounds two different things, and it pays to bound both:
  - `max_runtime` — how long an attempt may execute
  - `max_queued_time` — how long it may **wait for capacity** before Flyte gives up.
    This is your automatic detector for "the cluster has no free capacity" — instead of a
    run silently pending for hours, you get a clear, attributable failure.

> 💬 **Discuss:** what should `max_queued_time` be for this customer's heaviest jobs?
> Which events in their environment (spot exhaustion, quota, autoscaler limits) would
> trip it — and who should hear about it when it fires?

In [ ]:
from datetime import timedelta

from flyte import Timeout


@env.task(
    retries=3,
    timeout=Timeout(
        max_runtime=timedelta(minutes=10),
        max_queued_time=timedelta(minutes=5),
    ),
)
async def call_moderation_api(review: dict) -> dict:
    import random
    if random.random() < 0.2:
        raise ConnectionError(f"transient failure moderating review {review['id']}")
    return {**review, "moderation": "clean"}

In [ ]:
# retries=3 recovers from the transient failures — some actions show a retry
# before succeeding (visible in the run timeline).
@env.task
async def moderate_all(reviews: List[dict]) -> List[dict]:
    out: List[dict] = []
    async for r in flyte.map.aio(call_moderation_api, reviews, return_exceptions=True):
        if isinstance(r, Exception):
            raise r
        out.append(r)
    return out


run = flyte.run(moderate_all, reviews=SAMPLE_REVIEWS)
print(run.url)
run.wait()
run.outputs()

## 4. Recovering from OOM with more memory

The classic failure in data pipelines: input sizes vary 100×, so either you over-provision
everything or some shards OOM. In v2 the *pipeline itself* can catch
`flyte.errors.OOMError` and re-invoke with bigger resources — pay for the big pod only
when a skewed shard needs it:

In [ ]:
import flyte.errors


@env.task
async def build_shard_index(reviews_in_shard: int) -> int:
    # allocate ~1kB per review to simulate size-dependent memory use
    data = [b"x" * 1024 for _ in range(reviews_in_shard)]
    return len(data)


@env.task
async def resilient_index(reviews_in_shard: int) -> int:
    try:
        return await build_shard_index(reviews_in_shard=reviews_in_shard)
    except flyte.errors.OOMError:
        print("OOM at 1Gi — retrying this shard with 4Gi")
        return await build_shard_index.override(
            resources=flyte.Resources(cpu="1", memory="4Gi")
        )(reviews_in_shard=reviews_in_shard)

In [ ]:
# Happy path: 100k "reviews" (~100MB) fits the 1Gi box and runs straight through.
run = flyte.run(resilient_index, reviews_in_shard=100_000)
print(run.url)
run.wait()
print(run.outputs())

# To WATCH the recovery: 2M reviews (~2GB) OOMs the 1Gi first attempt; the except
# branch retries at 4Gi and succeeds (~1-2 min; the first attempt fails on purpose).
# run = flyte.run(resilient_index, reviews_in_shard=2_000_000)
# print(run.url); run.wait(); print(run.outputs())

This composes with `retries`: transient infrastructure failures use the retry budget,
while the OOM path *changes the resources* rather than retrying the same doomed
configuration. In operational reviews this is the cleanest way to separate **platform
errors** (node lost, image pull) from **user-code errors** (OOM at any size, exceptions)
— more in [appendix B](./appendix/B-observability-and-debugging.md).

## 5. Checkpointing inside a task: `@flyte.trace`

Tasks are the unit of *retry*, but long tasks often have expensive internal phases. Our
moderation pass is exactly that shape: **submit** a batch to the external API, then
**wait** for it to finish. `@flyte.trace` on an `async` helper makes each call a
**checkpoint**: on retry, completed calls are replayed from the record instead of
re-executed — no double-submissions, and each call shows up in the UI timeline.

In [ ]:
@flyte.trace
async def submit_moderation_batch(ids: List[int]) -> str:
    # e.g. POST /batches to the moderation vendor; returns a job handle
    await asyncio.sleep(0.1)
    return f"job-{min(ids)}-{max(ids)}"


@flyte.trace
async def wait_for_moderation(job_id: str) -> int:
    # e.g. poll until the vendor finishes
    await asyncio.sleep(0.2)
    return len(job_id)


@env.task(retries=3)
async def moderate_corpus(review_ids: List[int]) -> int:
    job = await submit_moderation_batch(review_ids)     # checkpoint 1
    return await wait_for_moderation(job)               # checkpoint 2

In [ ]:
# The traced submit/wait helpers show up as steps inside the task's timeline.
run = flyte.run(moderate_corpus, review_ids=[r["id"] for r in SAMPLE_REVIEWS])
print(run.url)
run.wait()
run.outputs()

If the pod dies while polling, the retry **skips re-submission** — the submit result is
replayed from the trace. Rules of thumb:

- Only `async def` helpers can be traced
- Trace non-deterministic/external calls; don't trace cheap pure functions
- Keep traced inputs/outputs small and serializable

## 6. Signaling: gate the dataset promotion

The processed corpus is about to become the input for feature engineering and training.
`flyte.new_condition` pauses the pipeline until a human signs off on promoting it — a
first-class condition action, no polling loops:

In [ ]:
from datetime import timedelta

import flyte.errors


@env.task
async def promote_dataset(summary: str) -> str:
    approval = await flyte.new_condition.aio(
        "promote_processed_corpus",
        prompt=f"Promote this processed corpus for training? {summary}",
        data_type=bool,
        timeout=timedelta(hours=4),
    )
    try:
        approved = await approval.wait.aio()
    except flyte.errors.ConditionTimedoutError:
        return "expired: nobody approved within 4h"

    return "promoted" if approved else "rejected"

In [ ]:
# Submit the gate — it PAUSES at the condition and hands control back immediately
# (flyte.run submits; it does not block on the human).
gate = flyte.run(promote_dataset, summary="6 reviews processed, 2 flagged negative")
print(gate.url)          # open it: the run is waiting for the signal
print("run name:", gate.name)

In [ ]:
# Approve it (run this once the UI shows the run waiting for the signal).
import flyte.remote

cond = flyte.remote.Condition.get(
    "promote_processed_corpus", run_name=gate.name, action_name="promote_dataset"
)
cond.signal(True)
gate.wait()
gate.outputs()           # -> "promoted"

While the run is paused you can signal from the UI, the CLI, or Python:

```bash
flyte get condition <run-name>
flyte signal condition <run-name> promote_processed_corpus true
```

```python
import flyte.remote
cond = flyte.remote.Condition.get("promote_processed_corpus", run_name="...", action_name="promote_dataset")
cond.signal(True)
```

`data_type` can be `bool`, `int`, `float`, or `str` — a condition can also *collect* a
typed value (a threshold, a promotion note) and feed it back into the workflow.

**Story checkpoint:** the corpus is enriched in parallel, moderation is checkpointed
against retries, and promotion has a human gate. Next problem: we'll re-run this pipeline
dozens of times while iterating on features — without paying for the same work twice.

## Further reading

- Union docs: [Build tasks](https://www.union.ai/docs/v2/union/user-guide/task-programming/) (fan-out) · [Traces](https://www.union.ai/docs/v2/union/user-guide/task-programming/traces/) · [External conditions](https://www.union.ai/docs/v2/union/user-guide/task-programming/conditions/)
- Next: [04-caching-and-reproducibility](./04-caching-and-reproducibility.ipynb)
- The pattern at 100k+ scale: [05-reusable-containers](./05-reusable-containers.ipynb)